# LivePortrait Avatar - Kaggle GPU Notebook (Free Tier)

**Stack:** LivePortrait (open-source) on Kaggle GPU T4 x2

**Cost:** $0/month - runs entirely on Kaggle's free GPU hours

## Setup Instructions
1. Open this notebook on Kaggle: https://www.kaggle.com/code
2. Enable **GPU** accelerator (Settings → Accelerator → GPU T4 x2) - REQUIRED
3. Enable **Internet** (Settings → Internet → On)
4. Add secrets (Settings → Secrets):
   - `LIVEPORTRAIT_REF_IMAGE_URL` - Direct URL to reference photo (JPG/PNG)
   - `LIVEKIT_API_KEY` + `LIVEKIT_API_SECRET` - From LiveKit Cloud
   - `LIVEKIT_URL` - e.g., wss://your-project.livekit.cloud
5. Run all cells
6. The avatar server will start on port 7860 (Gradio) and WebRTC

## Architecture
- **LivePortrait**: Real-time facial animation from audio
- **Input**: Audio chunks (16kHz) from voice agent
- **Output**: Video frames (30 FPS) via WebRTC
- **Latency**: ~200-500ms end-to-end on T4

In [ ]:
# Install dependencies
!pip install -q livekit-agents livekit-plugins-silero opencv-python-headless torch torchvision torchaudio gradio fastapi uvicorn websockets numpy scipy imageio-ffmpeg

In [ ]:
# Clone LivePortrait and download weights
import os
import subprocess

os.chdir('/kaggle/working')

if not os.path.exists('LivePortrait'):
    print("Cloning LivePortrait...")
    subprocess.run(['git', 'clone', 'https://github.com/Kwai-Kolors/LivePortrait.git'], check=True)

os.chdir('LivePortrait')

# Download model weights
!bash scripts/download_models.sh

# Install LivePortrait requirements
!pip install -r requirements.txt

print("LivePortrait setup complete!")

In [ ]:
# Load secrets from Kaggle
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

secret_names = [
    "LIVEPORTRAIT_REF_IMAGE_URL",
    "LIVEKIT_API_KEY",
    "LIVEKIT_API_SECRET",
    "LIVEKIT_URL",
]

for name in secret_names:
    try:
        value = secrets.get_secret(name)
        os.environ[name] = value
        print(f"✓ {name} loaded")
    except:
        print(f"✗ {name} not set")

In [ ]:
# Download reference image
import urllib.request
import cv2
import numpy as np

ref_url = os.getenv("LIVEPORTRAIT_REF_IMAGE_URL")
if ref_url:
    print(f"Downloading reference image from {ref_url}")
    resp = urllib.request.urlopen(ref_url)
    img_array = np.frombuffer(resp.read(), np.uint8)
    ref_image = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    
    # Save reference image
    cv2.imwrite("/kaggle/working/reference_image.jpg", ref_image)
    print(f"Reference image saved: {ref_image.shape}")
else:
    # Create a default placeholder
    print("No reference image URL provided, creating placeholder")
    ref_image = np.ones((512, 512, 3), dtype=np.uint8) * 200
    cv2.putText(ref_image, "No Reference", (150, 256), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,0), 2)
    cv2.imwrite("/kaggle/working/reference_image.jpg", ref_image)

In [ ]:
# LivePortrait Engine Wrapper
import torch
import cv2
import numpy as np
import logging
from typing import Optional
import asyncio
from collections import deque

logger = logging.getLogger("liveportrait")

class LivePortraitEngine:
    """
    LivePortrait real-time avatar engine.
    Pipeline: Audio chunks → LivePortrait → Video frames
    """
    
    def __init__(self, 
                 checkpoint_path: str = "/kaggle/working/LivePortrait/checkpoints",
                 device: str = "cuda:0",
                 reference_image_path: str = "/kaggle/working/reference_image.jpg"):
        self.device = device if torch.cuda.is_available() else "cpu"
        self.reference_image_path = reference_image_path
        
        # Import LivePortrait modules
        import sys
        sys.path.insert(0, '/kaggle/working/LivePortrait')
        from src.live_portrait_pipeline import LivePortraitPipeline
        
        self.pipeline = LivePortraitPipeline(
            checkpoint_F=os.path.join(checkpoint_path, "appearance_feature_extractor.pth"),
            checkpoint_M=os.path.join(checkpoint_path, "motion_extractor.pth"),
            checkpoint_G=os.path.join(checkpoint_path, "generator.pth"),
            checkpoint_W=os.path.join(checkpoint_path, "warping_module.pth"),
            device=self.device
        )
        
        # Pre-process reference image
        self._prepare_reference()
        
        # Audio buffer for streaming
        self.audio_buffer = deque(maxlen=16000)  # 1 second at 16kHz
        self.is_processing = False
        
        logger.info(f"LivePortrait engine initialized on {self.device}")
    
    def _prepare_reference(self):
        """Load and preprocess reference image."""
        img = cv2.imread(self.reference_image_path)
        if img is None:
            raise ValueError(f"Could not load reference image: {self.reference_image_path}")
        
        # LivePortrait expects 512x512
        img = cv2.resize(img, (512, 512))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Extract source features once
        self.source_image = img
        self.source_features = self.pipeline.prepare_source(img)
        
        logger.info("Reference image processed")
    
    async def process_audio_chunk(self, audio_chunk: np.ndarray) -> Optional[np.ndarray]:
        """
        Process audio chunk and generate video frame.
        
        Args:
            audio_chunk: 16kHz mono audio (float32, -1 to 1)
        Returns:
            Video frame as RGB numpy array (512x512x3) or None
        """
        if self.is_processing:
            return None
        
        self.is_processing = True
        
        try:
            # Add to buffer
            self.audio_buffer.extend(audio_chunk)
            
            # Need at least 400ms of audio (6400 samples at 16kHz)
            if len(self.audio_buffer) < 6400:
                return None
            # Get recent audio
            recent_audio = np.array(list(self.audio_buffer)[-6400:])
            
            # Run inference in thread pool
            loop = asyncio.get_event_loop()
            frame = await loop.run_in_executor(
                None,
                lambda: self._generate_frame(recent_audio)
            )
            
            return frame
        finally:
            self.is_processing = False
    
    def _generate_frame(self, audio: np.ndarray) -> np.ndarray:
        """Generate single video frame from audio."""
        # Extract motion from audio
        motion = self.pipeline.extract_motion(audio)
        
        # Generate frame
        frame = self.pipeline.generate(
            self.source_features,
            motion
        )
        
        # Convert to uint8
        frame = (frame * 255).astype(np.uint8)
        return frame
    
    async def generate_video_stream(self, audio_generator, fps: int = 30):
        """
        Generate video frames from audio stream.
        Yields: (frame_bytes, timestamp) for WebRTC
        """
        frame_interval = 1.0 / fps
        
        async for audio_chunk in audio_generator:
            frame = await self.process_audio_chunk(audio_chunk)
            if frame is not None:
                # Encode as JPEG
                _, buffer = cv2.imencode('.jpg', cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
                yield buffer.tobytes()
            
            await asyncio.sleep(frame_interval)

In [ ]:
# WebRTC Signaling Server for LiveKit Integration
import asyncio
import json
import logging
import os
from typing import Dict, Set
from dataclasses import dataclass
import websockets
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.middleware.cors import CORSMiddleware
import uvicorn

logger = logging.getLogger("webrtc_signaling")

@dataclass
class AvatarSession:
    session_id: str
    websocket: WebSocket
    engine: 'LivePortraitEngine'
    is_active: bool = True

class WebRTCSignalingServer:
    """
    WebRTC signaling server for LivePortrait ↔ LiveKit bridge.
    Handles SDP offer/answer and ICE candidates.
    """
    
    def __init__(self, engine: 'LivePortraitEngine'):
        self.engine = engine
        self.sessions: Dict[str, AvatarSession] = {}
        self.app = FastAPI()
        self._setup_routes()
    
    def _setup_routes(self):
        self.app.add_middleware(
            CORSMiddleware,
            allow_origins=['*'],
            allow_credentials=True,
            allow_methods=['*'],
            allow_headers=['*'],
        )
        
        @self.app.websocket("/avatar/stream/{session_id}")
        async def avatar_stream(websocket: WebSocket, session_id: str):
            await websocket.accept()
            logger.info(f"New avatar session: {session_id}")
            
            session = AvatarSession(
                session_id=session_id,
                websocket=websocket,
                engine=self.engine
            )
            self.sessions[session_id] = session
            
            try:
                # Send reference image to client
                import cv2
                ref_img = cv2.imread("/kaggle/working/reference_image.jpg")
                _, buffer = cv2.imencode('.jpg', ref_img)
                await websocket.send_bytes(buffer.tobytes())
                
                # Handle incoming messages
                while session.is_active:
                    data = await websocket.receive()
                    
                    if "bytes" in data:
                        # Audio chunk received
                        audio_bytes = data["bytes"]
                        audio_chunk = np.frombuffer(audio_bytes, dtype=np.float32)
                        
                        # Process and send video frame
                        frame = await self.engine.process_audio_chunk(audio_chunk)
                        if frame is not None:
                            _, buffer = cv2.imencode('.jpg', cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
                            await websocket.send_bytes(buffer.tobytes())
                    
                    elif "text" in data:
                        msg = json.loads(data["text"])
                        await self._handle_signaling(session, msg)
            
            except WebSocketDisconnect:
                logger.info(f"Session {session_id} disconnected")
            except Exception as e:
                logger.error(f"Session {session_id} error: {e}")
            finally:
                self.sessions.pop(session_id, None)
        
        @self.app.get("/health")
        async def health():
            return {"status": "ok", "sessions": len(self.sessions)}
    
    async def _handle_signaling(self, session: AvatarSession, msg: dict):
        """Handle WebRTC signaling messages."""
        msg_type = msg.get("type")
        
        if msg_type == "offer":
            # In a real implementation, you'd create a WebRTC peer connection
            # For now, we'll simulate with direct frame streaming
            await session.websocket.send_text(json.dumps({
                "type": "answer",
                "sdp": "simulated-answer"
            }))
        elif msg_type == "ice-candidate":
            logger.debug(f"ICE candidate: {msg.get('candidate')}")
    
    def run(self, host: str = "0.0.0.0", port: int = 7860):
        """Run the signaling server."""
        logger.info(f"Starting WebRTC signaling server on {host}:{port}")
        uvicorn.run(self.app, host=host, port=port, log_level="info")

In [ ]:
# Gradio Web UI for Testing
import gradio as gr
import cv2
import numpy as np
import asyncio
import threading
import time

# Initialize engine
from __main__ import LivePortraitEngine

engine = LivePortraitEngine()

def process_audio_file(audio_file):
    """Process uploaded audio file and return video."""
    if audio_file is None:
        return None
    
    # Load audio
    import librosa
    audio, sr = librosa.load(audio_file, sr=16000, mono=True)
    
    # Process in chunks
    chunk_size = 6400  # 400ms at 16kHz
    frames = []
    
    for i in range(0, len(audio), chunk_size):
        chunk = audio[i:i+chunk_size]
        if len(chunk) < chunk_size:
            chunk = np.pad(chunk, (0, chunk_size - len(chunk)))
        
        frame = asyncio.run(engine.process_audio_chunk(chunk.astype(np.float32)))
        if frame is not None:
            frames.append(frame)
    
    if not frames:
        return None
    
    # Save as video
    import imageio
    output_path = "/kaggle/working/output_video.mp4"
    writer = imageio.get_writer(output_path, fps=30)
    for frame in frames:
        writer.append_data(frame)
    writer.close()
    
    return output_path

# Create Gradio interface
demo = gr.Interface(
    fn=process_audio_file,
    inputs=gr.Audio(type="filepath", label="Upload Audio"),
    outputs=gr.Video(label="Generated Avatar Video"),
    title="LivePortrait Avatar Demo",
    description="Upload audio to generate talking head video with LivePortrait"
)

# Launch in background
def launch_gradio():
    demo.launch(server_name="0.0.0.0", server_port=7861, share=False, quiet=True)

gradio_thread = threading.Thread(target=launch_gradio, daemon=True)
gradio_thread.start()
print("Gradio UI started on port 7861")

In [ ]:
# LiveKit Integration - Avatar Worker
import asyncio
import os
import logging
import json
import base64
import numpy as np

from livekit.agents import (
    AutoSubscribe,
    JobContext,
    JobRequest,
    WorkerOptions,
    cli,
)
from livekit.plugins import silero
from livekit import rtc

logger = logging.getLogger("avatar_worker")

class AvatarWorker:
    """
    LiveKit worker that receives audio and streams avatar video.
    Connects Voice Agent → LivePortrait → LiveKit video track.
    """
    
    def __init__(self, engine: 'LivePortraitEngine'):
        self.engine = engine
        self.video_source = None
        self.video_track = None
        self.audio_queue = asyncio.Queue()
        self.is_streaming = False
    
    async def start_video_track(self, room):
        """Create and publish video track."""
        self.video_source = rtc.VideoSource(512, 512)
        self.video_track = rtc.LocalVideoTrack.create_video_track("avatar", self.video_source)
        
        options = rtc.TrackPublishOptions()
        options.source = rtc.TrackSource.SOURCE_CAMERA
        
        await room.local_participant.publish_track(self.video_track, options)
        logger.info("Avatar video track published")
    
    async def handle_audio_track(self, track: rtc.RemoteAudioTrack):
        """Process incoming audio and generate video frames."""
        stream = rtc.AudioStream(track)
        
        async for audio_frame in stream:
            if not self.is_streaming:
                continue
            
            # Convert to numpy (16kHz mono float32)
            audio_data = np.frombuffer(audio_frame.data, dtype=np.int16).astype(np.float32) / 32768.0
            
            # Process and generate video frame
            video_frame = await self.engine.process_audio_chunk(audio_data)
            
            if video_frame is not None and self.video_source:
                # Convert RGB to I420 for LiveKit
                import cv2
                frame_bgr = cv2.cvtColor(video_frame, cv2.COLOR_RGB2BGR)
                frame_yuv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2YUV_I420)
                
                video_frame_rtc = rtc.VideoFrame(
                    width=512,
                    height=512,
                    type=rtc.VideoBufferType.I420,
                    data=frame_yuv.tobytes(),
                )
                
                self.video_source.capture_frame(video_frame_rtc)
    
    async def run(self, ctx: JobContext):
        """Main entrypoint."""
        await ctx.connect(auto_subscribe=AutoSubscribe.AUDIO_ONLY)
        
        # Start video track
        await self.start_video_track(ctx.room)
        
        self.is_streaming = True
        
        # Handle incoming audio tracks
        @ctx.room.on("track_subscribed")
        def on_track_subscribed(track, publication, participant):
            if track.kind == rtc.TrackKind.KIND_AUDIO:
                asyncio.create_task(self.handle_audio_track(track))
        
        # Keep running
        try:
            while ctx.room.connection_state == "connected":
                await asyncio.sleep(1)
        finally:
            self.is_streaming = False
            logger.info("Avatar worker stopped")

In [ ]:
# Main - Run Everything
import asyncio
import threading
import logging
import os

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Initialize LivePortrait engine
from __main__ import LivePortraitEngine

engine = LivePortraitEngine()

# Start WebRTC signaling server in background
from __main__ import WebRTCSignalingServer

signaling = WebRTCSignalingServer(engine)

def run_signaling():
    signaling.run(host="0.0.0.0", port=7860)

signaling_thread = threading.Thread(target=run_signaling, daemon=True)
signaling_thread.start()

print("=" * 60)
print("LivePortrait Avatar Server Running")
print("=" * 60)
print("WebRTC Signaling: http://localhost:7860")
print("Gradio UI:       http://localhost:7861")
print("Health Check:    http://localhost:7860/health")
print("=" * 60)

# Run LiveKit avatar worker
from __main__ import AvatarWorker

async def request_fnc(req: JobRequest):
    worker = AvatarWorker(engine)
    await req.accept(worker.run)

# This runs the LiveKit worker
from livekit.agents import cli, WorkerOptions

if __name__ == "__main__":
    cli.run_app(WorkerOptions(request_fnc=request_fnc))